In [1]:
!git clone https://github.com/sara-graca/acoustic-vs-neural-representations.git
%cd acoustic-vs-neural-representations
!pip install tgt chardet

Cloning into 'acoustic-vs-neural-representations'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 95 (delta 36), reused 87 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 72.57 KiB | 2.59 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/kaggle/working/acoustic-vs-neural-representations
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 1.4 MB/s eta 0:00:00


In [2]:
import shutil
import os

os.makedirs("data/parsed", exist_ok=True)
os.makedirs("data/features", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)

shutil.copy(
    "/kaggle/input/datasets/sara2graca/statistics-parsed-data/phonemes.csv",
    "data/parsed/phonemes.csv"
)

shutil.copytree(
    "/kaggle/input/datasets/sara2graca/statistics-all-data/ru-fr_interference",
    "data/raw/ru-fr_interference"
)

print("Data ready")

Data ready


In [3]:
import os
import yaml
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm
import librosa
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model

df = pd.read_csv("data/parsed/phonemes.csv")
df["wav_path"] = df["wav_path"].apply(
    lambda p: os.path.join(
        "/kaggle/input/datasets/sara2graca/statistics-all-data",
        p.replace("data\\raw\\", "").replace("data/raw/", "").replace("\\", "/")
    )
)

device = "cuda" if torch.cuda.is_available() else "cpu"

XLSR_MODEL = "facebook/wav2vec2-large-xlsr-53"
XLSR_LAYER  = 20
XLSR_OUTPUT = "data/features/features_xlsr_layer20.npz"

print(f"Loading {XLSR_MODEL}...")

xlsr_processor = Wav2Vec2FeatureExtractor.from_pretrained(XLSR_MODEL)
xlsr_model     = Wav2Vec2Model.from_pretrained(XLSR_MODEL, output_hidden_states=True)
xlsr_model     = xlsr_model.to(device)
xlsr_model.eval()

xlsr_embeddings = {}
current_wav, current_audio, current_sr = None, None, None

for i, row in tqdm(df.iterrows(), total=len(df)):
    if row["wav_path"] != current_wav:
        current_wav = row["wav_path"]
        try:
            current_audio, current_sr = librosa.load(current_wav, sr=16000)
        except Exception:
            current_audio, current_sr = None, None

    if current_audio is None:
        xlsr_embeddings[i] = np.zeros(xlsr_model.config.hidden_size)
        continue

    onset_sample  = int(row["onset"] * current_sr)
    offset_sample = int(row["offset"] * current_sr)
    segment = current_audio[onset_sample:offset_sample]

    if len(segment) < 400:  # ~25ms at 16000 Hz
        xlsr_embeddings[i] = np.zeros(xlsr_model.config.hidden_size)
        continue

    inputs = xlsr_processor(segment, sampling_rate=16000, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = xlsr_model(**inputs, output_hidden_states=True)

    hidden    = outputs.hidden_states[XLSR_LAYER]
    embedding = hidden[0].mean(dim=0).cpu().numpy()
    xlsr_embeddings[i] = embedding

np.savez(XLSR_OUTPUT, **{str(k): v for k, v in xlsr_embeddings.items()})
print(f"Done: {len(xlsr_embeddings)} embeddings → {XLSR_OUTPUT}")

shutil.copy("data/features/features_xlsr_layer20.npz", "/kaggle/working/features_xlsr_layer20.npz")
print("Saved")

Loading facebook/wav2vec2-large-xlsr-53...


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-xlsr-53
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/22919 [00:00<?, ?it/s]

Done: 22919 embeddings → data/features/features_xlsr_layer20.npz
Saved
